## Step 1 — Install dependencies

In [ ]:
!pip install -q --upgrade yfinance
print('✅ Dependencies ready')

## Step 2 — Imports & configuration

In [ ]:
import os, time, warnings, logging
from datetime import datetime

import numpy  as np
import pandas as pd
import yfinance as yf
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

OUTPUT_DIR   = '/kaggle/working'
RAW_OUT      = f'{OUTPUT_DIR}/training_raw.parquet'
FEAT_OUT     = f'{OUTPUT_DIR}/training_features.parquet'
LOG_OUT      = f'{OUTPUT_DIR}/collection_log.csv'

SLEEP_SEC    = 1.5    # polite delay between requests
MIN_QUARTERS = 4      # skip companies with fewer valid quarters after cleaning
MAX_QUARTERS = 10     # fetch up to 10 quarters — gives 6 valid YoY values

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)
print('✅ Config ready')

## Step 3 — Ticker universe (200+ companies)

In [ ]:
IT_SERVICES = [
    'EPAM','CTSH','ACN','INFY','WIT','GLOB','PEGA','EXLS','KFRC',
    'MMS','PRFT','LDOS','SAIC','BAH','MANT','ICFI','CLPS',
    'NICE','TTEC','NSIT','CDW','FORR','GCMG','IGATE','WPRK',
]

CLOUD_SAAS = [
    'NOW','CRM','WDAY','VEEV','FIVN','PCTY','PAYC',
    'BILL','BRZE','RNG','TOST','FRSH','APPF','ZEN',
    'NCNO','ALRM','SUMO','KVYO','BOX','DOCU','JAMF','GTLB',
]

ENTERPRISE_SOFTWARE = [
    'ORCL','SAP','MSFT','IBM','ANSS','PTC','AZPN','CDNS',
    'SNPS','MANH','SPSC','JKHY','SMAR','ALKT','LPSN','EVCM',
    'ADBE','INTU','PCOR','BSY','GLBE','COUP',
]

CYBERSECURITY = [
    'PANW','CRWD','FTNT','ZS','OKTA','S','TENB','CYBR',
    'VRNT','QLYS','CHKP','RPD','SAIL','SSTI','ATEN',
    'OSPN','SCWX','MNDT','ESTC',
]

DATA_ANALYTICS = [
    'SNOW','DDOG','MDB','CFLT','AI','BBAI','ASAN','MNDY',
    'MSTR','VERX','TASK','WEAV','RAMP','PLTR','DOMO',
]

HARDWARE_INFRA = [
    'NTAP','PSTG','HPE','DELL','STX','ANET','JNPR',
    'SMCI','LOGI','COHU','VIAVI','WDC','TRMB',
    'CGNX','TTMI','SANM',
]

TELECOM_IT = [
    'CSCO','ERIC','NOK','CIEN','INFN','VIAV',
    'CALX','LUMN','RBBN','DGII','IDCC','SATS',
]

FINTECH_IT = [
    'FISV','FIS','EVTC','WEX','SS','ACIW',
    'EVERI','QTWO','RPAY','PRGS','FLYW','REPAY',
]

SEMICONDUCTORS = [
    'NVDA','AMD','QCOM','MRVL','ONTO','CRUS',
    'AMAT','KLAC','LRCX','WOLF','SWKS','SLAB',
    'ENTG','FORM','ACLS','UCTT','DIOD',
]

CONSULTING_MGMT = [
    'G','HURN','MAXIMUS','CEVA','PSNL','KELYA',
    'PWSC','GDYN','CACI','BAH','ICF','LDOS',
]

ALL_TICKERS = list(dict.fromkeys(
    IT_SERVICES + CLOUD_SAAS + ENTERPRISE_SOFTWARE +
    CYBERSECURITY + DATA_ANALYTICS + HARDWARE_INFRA +
    TELECOM_IT + FINTECH_IT + SEMICONDUCTORS + CONSULTING_MGMT
))

print(f'✅ Universe: {len(ALL_TICKERS)} tickers across 10 sub-sectors')
for name, lst in [
    ('IT Services',         IT_SERVICES),
    ('Cloud / SaaS',        CLOUD_SAAS),
    ('Enterprise Software', ENTERPRISE_SOFTWARE),
    ('Cybersecurity',       CYBERSECURITY),
    ('Data & Analytics',    DATA_ANALYTICS),
    ('Hardware / Infra',    HARDWARE_INFRA),
    ('Telecom IT',          TELECOM_IT),
    ('Fintech IT',          FINTECH_IT),
    ('Semiconductors',      SEMICONDUCTORS),
    ('Consulting & Mgmt',   CONSULTING_MGMT),
]:
    print(f'  {name:<22} {len(lst)} tickers')

## Step 4 — Helper functions

In [ ]:
def safe_transpose(df):
    if df is None or df.empty:
        return pd.DataFrame()
    t = df.T.copy()
    t.index = pd.to_datetime(t.index)
    t.index.name = 'date'
    return t


def get_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(0.0, index=df.index)


def fetch_yfinance(ticker, retries=3):
    """Fetch with exponential backoff retry — handles API rate limits."""
    for attempt in range(retries):
        try:
            tk = yf.Ticker(ticker)
            # fetch_info=False speeds up the call — we get info separately
            income   = safe_transpose(tk.quarterly_financials)
            balance  = safe_transpose(tk.quarterly_balance_sheet)
            cashflow = safe_transpose(tk.quarterly_cashflow)
            try:
                info = tk.info or {}
            except Exception:
                info = {}
            return {'income': income, 'balance': balance, 'cashflow': cashflow, 'info': info}
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                log.warning(f'  {ticker} attempt {attempt+1} failed ({e}), retrying in {wait}s...')
                time.sleep(wait)
            else:
                raise e


def build_features(ticker, data):
    """Merge statements, compute ratios, apply correct clipping per feature."""
    inc  = data.get('income',   pd.DataFrame())
    bal  = data.get('balance',  pd.DataFrame())
    cf   = data.get('cashflow', pd.DataFrame())
    info = data.get('info',     {})

    if inc.empty and bal.empty:
        return pd.DataFrame()

    frames = [f for f in [inc, bal, cf] if not f.empty]
    base   = frames[0].copy()
    for f in frames[1:]:
        base = base.join(f, how='outer', rsuffix='_dup')
    base = base.loc[:, ~base.columns.str.endswith('_dup')].sort_index()

    # Keep latest MAX_QUARTERS rows
    base = base.tail(MAX_QUARTERS)

    df = pd.DataFrame(index=base.index)
    df['ticker']     = ticker
    df['sector']     = info.get('sector', 'Technology')
    df['market_cap'] = info.get('marketCap', None)

    # ── Raw financials ──────────────────────────────────────────────
    df['revenue']             = get_col(base, 'Total Revenue', 'Revenue')
    df['gross_profit']        = get_col(base, 'Gross Profit')
    df['operating_income']    = get_col(base, 'Operating Income', 'EBIT')
    df['net_income']          = get_col(base, 'Net Income')
    df['interest_expense']    = get_col(base, 'Interest Expense').abs()
    df['total_assets']        = get_col(base, 'Total Assets')
    df['total_liabilities']   = get_col(base, 'Total Liabilities Net Minority Interest', 'Total Liab')
    df['total_equity']        = get_col(base, 'Stockholders Equity', 'Total Stockholders Equity')
    df['current_assets']      = get_col(base, 'Current Assets')
    df['current_liabilities'] = get_col(base, 'Current Liabilities')
    df['total_debt']          = get_col(base, 'Total Debt', 'Long Term Debt')
    df['cash']                = get_col(base, 'Cash And Cash Equivalents',
                                               'Cash Cash Equivalents And Short Term Investments')
    df['operating_cash_flow'] = get_col(base, 'Operating Cash Flow',
                                               'Cash Flow From Continuing Operating Activities')
    df['capex']               = get_col(base, 'Capital Expenditure').abs()
    df['free_cash_flow']      = df['operating_cash_flow'] - df['capex']

    # Drop rows with missing or near-zero revenue (unusable)
    df = df.dropna(subset=['revenue'])
    df = df[df['revenue'].abs() > 1_000]   # > $1K (units are in dollars from yfinance)
    if df.empty:
        return pd.DataFrame()

    # ── Computed ratios ─────────────────────────────────────────────
    eps = 1e-9
    rev = df['revenue'].abs() + eps

    df['gross_margin']     = (df['gross_profit']     / rev)
    df['operating_margin'] = (df['operating_income'] / rev)
    df['net_margin']       = (df['net_income']        / rev)
    df['fcf_margin']       = (df['free_cash_flow']    / rev)
    df['roe']              = (df['net_income']        / (df['total_equity'].abs() + eps))
    df['roa']              = (df['net_income']        / (df['total_assets'].abs()  + eps))
    df['asset_turnover']   = (df['revenue']            / (df['total_assets'].abs()  + eps))
    df['current_ratio']    = (df['current_assets']    / (df['current_liabilities'].abs() + eps))

    # D/E: negative equity (e.g. DELL buybacks) → cap at 10 instead of producing D/E=79
    equity_positive = df['total_equity'].apply(lambda x: max(x, eps))
    df['debt_to_equity'] = (df['total_debt'] / equity_positive)

    # Interest coverage: cap at ±20 — beyond 20x means zero debt risk, no finer distinction needed
    df['interest_coverage'] = (df['operating_income'] / (df['interest_expense'] + eps))

    # YoY: with MAX_QUARTERS=10, periods=4 gives 6 valid values per company
    df['revenue_growth_yoy'] = df['revenue'].pct_change(periods=4)

    # ── Per-feature clipping (not all at ±100) ──────────────────────
    df['gross_margin']       = df['gross_margin'].clip(-2, 2)
    df['operating_margin']   = df['operating_margin'].clip(-2, 2)
    df['net_margin']         = df['net_margin'].clip(-2, 2)
    df['fcf_margin']         = df['fcf_margin'].clip(-2, 2)
    df['roe']                = df['roe'].clip(-5, 5)
    df['roa']                = df['roa'].clip(-1, 1)
    df['asset_turnover']     = df['asset_turnover'].clip(0, 5)
    df['current_ratio']      = df['current_ratio'].clip(0, 15)
    df['debt_to_equity']     = df['debt_to_equity'].clip(0, 10)   # caps DELL/STX issue
    df['interest_coverage']  = df['interest_coverage'].clip(-20, 20)
    df['revenue_growth_yoy'] = df['revenue_growth_yoy'].clip(-1, 3)

    return df.reset_index(drop=True)


print('✅ Helper functions ready')

## Step 5 — Run data collection
Fetches all companies with retry logic. Expect **8–15 minutes** for 200+ tickers.
Rate limit protection: 1.5s sleep between requests + exponential backoff on failure.

In [ ]:
all_frames = []
log_rows   = []
start_time = datetime.now()

for ticker in tqdm(ALL_TICKERS, desc='Collecting financials'):
    try:
        data     = fetch_yfinance(ticker)
        features = build_features(ticker, data)

        if features.empty or len(features) < MIN_QUARTERS:
            log_rows.append({'ticker': ticker, 'status': 'skipped',
                             'rows': len(features) if not features.empty else 0,
                             'reason': 'insufficient quarters'})
            time.sleep(SLEEP_SEC)
            continue

        all_frames.append(features)
        log_rows.append({'ticker': ticker, 'status': 'ok', 'rows': len(features), 'reason': ''})
        log.info(f'  {ticker}: {len(features)} quarters collected')

    except Exception as e:
        log_rows.append({'ticker': ticker, 'status': 'error', 'rows': 0, 'reason': str(e)[:120]})
        log.error(f'  {ticker}: FAILED — {e}')

    time.sleep(SLEEP_SEC)

elapsed = (datetime.now() - start_time).seconds
print(f'\n✅ Collection done in {elapsed}s')
print(f'   Collected : {len(all_frames)} companies')
print(f'   Skipped   : {sum(1 for r in log_rows if r["status"]=="skipped")} companies')
print(f'   Errors    : {sum(1 for r in log_rows if r["status"]=="error")} companies')

# Show what failed so you can investigate
failed = [r for r in log_rows if r['status'] != 'ok']
if failed:
    print('\n── Failed/Skipped tickers:')
    for r in failed:
        print(f'  {r["ticker"]:<10} [{r["status"]}] {r["reason"]}')

## Step 6 — Combine & save raw data

In [ ]:
raw = pd.concat(all_frames, axis=0)
raw = raw.sort_values(['ticker', 'date']).reset_index(drop=True)

raw.to_parquet(RAW_OUT, index=False)
pd.DataFrame(log_rows).to_csv(LOG_OUT, index=False)

print(f'✅ Raw data: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
print(f'   Companies : {raw["ticker"].nunique()}')
print(f'   Date range: {raw["date"].min().date()} → {raw["date"].max().date()}')
print(f'   Avg quarters/company: {raw.groupby("ticker").size().mean():.1f}')

## Step 7 — Fill missing values properly
Fill per-ticker → per-sector → global. Preserves company-specific patterns.

In [ ]:
RATIO_COLS = [
    'gross_margin','operating_margin','net_margin','fcf_margin',
    'roe','roa','debt_to_equity','current_ratio',
    'interest_coverage','asset_turnover','revenue_growth_yoy',
]

# revenue_growth_yoy will be NaN for first 4 rows of each company
# Fill with per-ticker median (same company's growth rate in other quarters)
# then sector median, then global median
print('Filling missing values...')
for col in RATIO_COLS:
    before = raw[col].isna().sum()
    raw[col] = raw.groupby('ticker')[col].transform(lambda x: x.fillna(x.median()))
    raw[col] = raw.groupby('sector')[col].transform(lambda x: x.fillna(x.median()))
    raw[col] = raw[col].fillna(raw[col].median())
    after  = raw[col].isna().sum()
    if before > 0:
        print(f'  {col:<25}: {before} → {after} missing')

print(f'\nTotal missing after fill: {raw[RATIO_COLS].isna().sum().sum()}')

## Step 8 — Create risk labels
Improved scoring adds interest_coverage signal (was missing in v1).

In [ ]:
def create_risk_labels(df):
    df    = df.copy()
    score = pd.Series(0, index=df.index)

    # Liquidity (most predictive per SHAP analysis)
    score += (df['current_ratio']      < 1.0 ).astype(int) * 2
    score += (df['current_ratio']      < 0.7 ).astype(int) * 1   # extra for critical

    # Leverage
    score += (df['debt_to_equity']     > 2.0 ).astype(int) * 2
    score += (df['interest_coverage']  < 1.5 ).astype(int) * 2   # NEW: can't service debt

    # Profitability
    score += (df['operating_margin']   < 0.0 ).astype(int) * 2
    score += (df['net_margin']         < -0.1).astype(int) * 1   # extra if deeply negative

    # Growth & cash
    score += (df['revenue_growth_yoy'] < -0.1).astype(int) * 1
    score += (df['free_cash_flow']     < 0   ).astype(int) * 1

    df['risk_score'] = score
    df['risk_label'] = pd.cut(
        score,
        bins=[-1, 1, 3, 100],
        labels=['low_risk', 'medium_risk', 'high_risk']
    ).astype(str)

    return df

labeled = create_risk_labels(raw)

print('✅ Risk labels created')
dist  = labeled['risk_label'].value_counts()
total = len(labeled)
for label, count in dist.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<15} {count:>5} rows ({pct:.1f}%)  {bar}')

if (labeled['risk_label'] == 'high_risk').mean() < 0.03:
    print('\n⚠ WARNING: high_risk < 3% — consider lowering risk score threshold')
    print('  Change bins=[-1, 1, 3, 100] to bins=[-1, 1, 2, 100] to label more as high_risk')

## Step 9 — Save ML-ready features

In [ ]:
SAVE_COLS = [
    'ticker','date','sector','market_cap',
    'revenue','gross_profit','operating_income','net_income',
    'total_assets','total_liabilities','total_equity',
    'current_assets','current_liabilities','total_debt','cash',
    'operating_cash_flow','free_cash_flow',
    'gross_margin','operating_margin','net_margin','fcf_margin',
    'roe','roa','debt_to_equity','current_ratio',
    'interest_coverage','asset_turnover','revenue_growth_yoy',
    'risk_score','risk_label',
]

keep_cols   = [c for c in SAVE_COLS if c in labeled.columns]
features_df = labeled[keep_cols].copy()
features_df.to_parquet(FEAT_OUT, index=False)

print(f'✅ Saved {len(features_df):,} rows × {len(keep_cols)} columns → {FEAT_OUT}')
print(f'   Companies : {features_df["ticker"].nunique()}')
print(f'   Labels    : {features_df["risk_label"].value_counts().to_dict()}')

## Step 10 — Data quality report

In [ ]:
print('── Missing values ──────────────────────────────────────')
missing = features_df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('  ✅ No missing values')
else:
    print(missing.to_string())

print('\n── Feature statistics ──────────────────────────────────')
RATIO_COLS = [
    'gross_margin','operating_margin','net_margin','fcf_margin',
    'roe','roa','debt_to_equity','current_ratio',
    'interest_coverage','asset_turnover','revenue_growth_yoy',
]
print(features_df[RATIO_COLS].describe().round(3).to_string())

print('\n── Quarters per company ────────────────────────────────')
qpc = features_df.groupby('ticker').size()
print(f'  min={qpc.min()}  max={qpc.max()}  mean={qpc.mean():.1f}')
print(f'  Companies with < 6 quarters: {(qpc < 6).sum()} (may have limited YoY values)')

print('\n── Sector breakdown ────────────────────────────────────')
print(features_df.groupby('sector')['ticker'].nunique().sort_values(ascending=False).to_string())

print('\n── Collection log summary ──────────────────────────────')
log_df = pd.read_csv(LOG_OUT)
print(log_df['status'].value_counts().to_string())
failed = log_df[log_df['status'] != 'ok']
if len(failed):
    print(f'\n  Failed tickers ({len(failed)}):')
    print(failed[['ticker','status','reason']].to_string(index=False))

## Step 11 — EPAM sanity check

In [ ]:
epam = features_df[features_df['ticker'] == 'EPAM'].copy()
print(f'EPAM quarters: {len(epam)}')
print(f'Date range   : {epam["date"].min().date()} → {epam["date"].max().date()}')
print(f'Labels       : {epam["risk_label"].value_counts().to_dict()}')
cols = ['date','revenue','operating_margin','current_ratio','debt_to_equity','revenue_growth_yoy','risk_label']
epam[cols].tail(8).round(3)

## ✅ Done

| File | Description |
|---|---|
| `training_raw.parquet` | Raw financials, all companies, all quarters |
| `training_features.parquet` | Cleaned ratios + risk labels, ML-ready |
| `collection_log.csv` | Which tickers succeeded / skipped / failed |

**Key improvements over v1:**
- 10 quarters per company (was 5) → 6 valid YoY values per company
- Retry logic with exponential backoff → fewer failures
- Per-feature clipping (not all ±100) → cleaner data
- Interest coverage added to risk scoring
- Missing values filled per-ticker/sector (not global median)